In [7]:
!pip install pandas numpy scikit-learn xgboost joblib matplotlib seaborn

In [8]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

import joblib

In [9]:
df = pd.read_csv("inpatientCharges.csv")

print(df.head())
print(df.info())

                             DRG Definition  Provider Id  \
0  039 - EXTRACRANIAL PROCEDURES W/O CC/MCC        10001   
1  039 - EXTRACRANIAL PROCEDURES W/O CC/MCC        10005   
2  039 - EXTRACRANIAL PROCEDURES W/O CC/MCC        10006   
3  039 - EXTRACRANIAL PROCEDURES W/O CC/MCC        10011   
4  039 - EXTRACRANIAL PROCEDURES W/O CC/MCC        10016   

                      Provider Name     Provider Street Address Provider City  \
0  SOUTHEAST ALABAMA MEDICAL CENTER      1108 ROSS CLARK CIRCLE        DOTHAN   
1     MARSHALL MEDICAL CENTER SOUTH  2505 U S HIGHWAY 431 NORTH          BOAZ   
2    ELIZA COFFEE MEMORIAL HOSPITAL          205 MARENGO STREET      FLORENCE   
3                 ST VINCENT'S EAST  50 MEDICAL PARK EAST DRIVE    BIRMINGHAM   
4     SHELBY BAPTIST MEDICAL CENTER     1000 FIRST STREET NORTH     ALABASTER   

  Provider State  Provider Zip Code Hospital Referral Region Description  \
0             AL              36301                          AL - Dothan   


In [10]:
df.columns = df.columns.str.strip().str.replace(" ", "_")
money_cols = [
    "Average_Covered_Charges",
    "Average_Total_Payments",
    "Average_Medicare_Payments"
]

for col in money_cols:
    df[col] = df[col].replace('[\$,]', '', regex=True).astype(float)

<>:9: SyntaxWarning: invalid escape sequence '\$'
<>:9: SyntaxWarning: invalid escape sequence '\$'
/tmp/ipykernel_979/4032989413.py:9: SyntaxWarning: invalid escape sequence '\$'
  df[col] = df[col].replace('[\$,]', '', regex=True).astype(float)


In [11]:
print(df.isnull().sum())

df.drop_duplicates(inplace=True)

DRG_Definition                          0
Provider_Id                             0
Provider_Name                           0
Provider_Street_Address                 0
Provider_City                           0
Provider_State                          0
Provider_Zip_Code                       0
Hospital_Referral_Region_Description    0
Total_Discharges                        0
Average_Covered_Charges                 0
Average_Total_Payments                  0
Average_Medicare_Payments               1
dtype: int64


In [12]:
def remove_outliers(df, column):

    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    return df[(df[column] >= lower) & (df[column] <= upper)]

df = remove_outliers(df, "Average_Covered_Charges")

In [13]:
df["charge_per_discharge"] = (
    df["Average_Covered_Charges"] / df["Total_Discharges"]
)

df["payment_gap"] = (
    df["Average_Covered_Charges"] - df["Average_Total_Payments"]
)

In [14]:
median_charge = df["Average_Covered_Charges"].median()

df["High_Cost"] = np.where(
    df["Average_Covered_Charges"] > median_charge,
    1,
    0
)

In [15]:
X = df.drop(
    columns=[
        "Provider_Name",
        "Provider_Id",
        "Average_Covered_Charges",
        "High_Cost"
    ]
)

y = df["High_Cost"]

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [17]:
categorical_cols = X.select_dtypes(include=["object"]).columns
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns

In [18]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

In [19]:
models = {

    "LogisticRegression": LogisticRegression(),

    "RandomForest": RandomForestClassifier(),

    "SVM": SVC(probability=True),

    "XGBoost": XGBClassifier(eval_metric="logloss")
}

In [20]:
param_grids = {

"LogisticRegression": {
    "model__C": [0.01, 0.1, 1, 10]
},

"RandomForest": {
    "model__n_estimators": [100, 200],
    "model__max_depth": [5, 10]
},

"SVM": {
    "model__C": [0.1, 1],
    "model__kernel": ["linear", "rbf"]
},

"XGBoost": {
    "model__n_estimators": [100, 200],
    "model__max_depth": [3, 6]
}

}

In [21]:
best_models = {}

for name in models:

    print(f"\nTraining {name}")

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", models[name])
    ])

    grid = GridSearchCV(
        pipeline,
        param_grids[name],
        cv=5,
        scoring="f1",
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    best_models[name] = grid.best_estimator_

    print("Best Params:", grid.best_params_)


Training LogisticRegression
Best Params: {'model__C': 1}

Training RandomForest
Best Params: {'model__max_depth': 10, 'model__n_estimators': 200}

Training SVM
Best Params: {'model__C': 1, 'model__kernel': 'linear'}

Training XGBoost
Best Params: {'model__max_depth': 6, 'model__n_estimators': 100}


In [22]:
results = {}

for name in best_models:

    model = best_models[name]

    preds = model.predict(X_test)

    results[name] = {

        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds),
        "Recall": recall_score(y_test, preds),
        "F1": f1_score(y_test, preds)
    }

print(pd.DataFrame(results).T)

                    Accuracy  Precision    Recall        F1
LogisticRegression  0.991109   0.992399  0.989890  0.991143
RandomForest        0.928450   0.952847  0.902275  0.926871
SVM                 0.990686   0.992392  0.989048  0.990717
XGBoost             0.996190   0.994958  0.997473  0.996214


In [23]:
best_model_name = max(
    results,
    key=lambda x: results[x]["F1"]
)

best_model = best_models[best_model_name]

print("Best Model:", best_model_name)

Best Model: XGBoost


In [24]:
probs = best_model.predict_proba(X_test)[:, 1]

roc = roc_auc_score(y_test, probs)

print("ROC-AUC:", roc)

ROC-AUC: 0.9999354711502267


In [25]:
if hasattr(best_model["model"], "feature_importances_"):

    importance = best_model["model"].feature_importances_

    print("Feature Importance:")

    print(importance[:20])

Feature Importance:
[0.00297458 0.00353366 0.10648919 0.00882257 0.00531474 0.7739235
 0.         0.         0.0022957  0.00525603 0.00592484 0.
 0.         0.         0.         0.         0.         0.
 0.         0.        ]


In [26]:
cv_score = cross_val_score(
    best_model,
    X,
    y,
    cv=5,
    scoring="f1"
)

print("CV F1 Score:", cv_score.mean())

CV F1 Score: 0.9754420667880801


In [27]:
joblib.dump(best_model, "best_hospital_model.pkl")

print("Model Saved Successfully")

Model Saved Successfully
